# NumCompute-Stream

Read the seeded synthetic CSV in 100-row chunks and compare buffers of 200 and 500.
Missing features are imputed. The data's decision boundary changes halfway through.


In [2]:
from pathlib import Path
import sys
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'numcompute_stream').exists() else Path.cwd().parent
if not (ROOT / 'numcompute_stream').exists():
    raise RuntimeError('Open this notebook from the repository root or demo directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from numcompute_stream import (Pipeline, SimpleImputer, StandardScaler,
    DecisionTreeClassifier, StreamingMetrics, StreamingStats)
from numcompute_stream.io import iter_csv_chunks
from numcompute_stream.visualise import (plot_metric_over_time, compare_models,
    plot_predictions_vs_ground_truth)

DATA = ROOT / 'demo' / 'data' / 'stream.csv'
print('Dataset:', DATA.name)


Dataset: stream.csv


## Train one chunk at a time

Each pipeline updates its imputer and scaler, transforms the chunk, and passes it
through the tree's public `partial_fit()` method. The tree retrains when its buffer
fills. We predict **before** training on each chunk; warm-up scores are NaN.

The scaler changes over time, so earlier buffered rows and current predictions
can use different scales. This simpler approach accepts scaling drift, which may
be substantial after a large distribution change. Retraining replaces the tree,
but does not rescale the rows already stored in its buffer.


In [ ]:
buffer_sizes = (200, 500)
models = [Pipeline([
    ('impute', SimpleImputer()),
    ('scale', StandardScaler()),
    ('tree', DecisionTreeClassifier(max_depth=5, buffer_size=size, random_state=2026)),
]) for size in buffer_sizes]
metrics = [StreamingMetrics() for _ in models]
histories = [[] for _ in models]
stats = StreamingStats(window_size=500, bins=np.linspace(-4, 4, 17))

for chunk_number, chunk in enumerate(iter_csv_chunks(
        DATA, chunk_size=100, skip_header=1, missing='propagate'), start=1):
    X, raw_y = chunk[:, :-1], chunk[:, -1]
    if not np.isfinite(raw_y).all() or not np.isin(raw_y, [0, 1]).all():
        raise ValueError('CSV labels must be finite binary values')
    y = raw_y.astype(int)
    stats.update_stats(X)
    for model, metric, history in zip(models, metrics, histories):
        chunk_accuracy = np.nan
        if model.is_fitted_:
            predictions = model.predict(X)
            chunk_accuracy = float(np.mean(predictions == y))
            metric.update(y, predictions)
        start = perf_counter()
        model.partial_fit(X, y)
        elapsed = perf_counter() - start
        history.append({'chunk': chunk_number, 'n_samples': len(y),
                        'chunk_accuracy': chunk_accuracy,
                        'cumulative_accuracy': metric.result()['accuracy'],
                        'fit_seconds': elapsed})


## Inspect the chunk logs

Untrained chunks do not count toward cumulative accuracy. Fit time includes
preprocessing, buffering and any tree retraining triggered by that chunk.
The command-line demo also uses `StreamTrainer` to log estimated model memory.


In [ ]:
for records in zip(*histories):
    print('Chunk', records[0]['chunk'], '|', ' | '.join(
        f"buffer={size}: accuracy={record['chunk_accuracy']:.3f}, "
        f"cumulative={record['cumulative_accuracy']:.3f}, fit={record['fit_seconds']:.4f}s"
        for size, record in zip(buffer_sizes, records)))

common = np.logical_and.reduce([
    np.isfinite([r['chunk_accuracy'] for r in history]) for history in histories])
for size, model, history in zip(buffer_sizes, models, histories):
    selected = [r for r, use in zip(history, common) if use]
    n = sum(r['n_samples'] for r in selected)
    score = sum(r['n_samples'] * r['chunk_accuracy'] for r in selected) / n if n else np.nan
    print('Buffer', size, '| common samples:', n, '| accuracy:', score,
          '| pending samples:', model.n_buffered_)


## Compare accuracy

The larger buffer warms up later. `common_accuracy` compares the same 1,100 scored
samples for both models. A final partial buffer stays pending.


In [ ]:
curves = [[record['chunk_accuracy'] for record in history] for history in histories]
fig, ax = plot_metric_over_time(curves[0], 'Test-before-train accuracy: buffer 200', 'Accuracy', show=True)
plt.close(fig)
fig, ax = compare_models(*curves, labels=['Buffer 200', 'Buffer 500'], show=True)
ax.set_ylabel('Test-before-train accuracy')
plt.close(fig)


## Inspect predictions and statistics

This plot uses the current tree after training. The logged metrics use predictions
made before training. Statistics describe the latest 500 rows.


In [ ]:
predictions = models[0].predict(X)
fig, ax = plot_predictions_vs_ground_truth(y, predictions, show=True)
plt.close(fig)
for size, metric in zip(buffer_sizes, metrics):
    print('Buffer', size, metric.result())

latest_stats = stats.result()
print('Latest 500 rows, valid counts:', latest_stats['count'])
print('Mean:', latest_stats['mean'])
print('Variance:', latest_stats['variance'])
print('Quartiles:', latest_stats['quantiles'])
print('Histogram shape:', latest_stats['histogram'].shape)


## Checks

Run `python -m pytest -q` and `MPLBACKEND=Agg python -m demo.check_notebook demo/stream_demo.ipynb`.
Reproduce the five-repeat benchmark with `python -m benchmark.stream_benchmark`.
It compares a threshold-by-threshold loop with the tree's vectorised split search,
then measures streaming buffers of 100, 200, 500 and 1000 rows. Both split-search
implementations must agree before timing. Results depend on the machine.
